## АХ: Агент-художник

Выполнили: Бутаков

Для начала, загрузим ключи из файла .env:

In [1]:
from dotenv import load_dotenv
import os

if load_dotenv(override=True):
    print(".env loaded")

folder_id = os.environ['folder_id']
api_key = os.environ['api_key']

.env loaded


## Шаг 1. Подключаемся к нейросети и делаем класс Agent

Сначала я создаю клиента, через которого буду общаться с нейросетью Яндекса. Дальше описываю класс `Agent`. Агент — это просто нейросеть с постоянной ролью (`instructions`) и памятью (`history`). Память нужна, потому что модель сама ничего не помнит, и ей приходится каждый раз отправлять всю переписку заново.

In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

model_uri = f"gpt://{folder_id}/deepseek-v4.1-flash"

class Agent:
    def __init__(self, name, instructions):
        self.name = name
        self.instructions = instructions
        self.history = []

    def say(self, message):
        self.history.append({"role": "user", "content": message})
        res = client.responses.create(
            model=model_uri,
            instructions=self.instructions,
            input=self.history,
        )
        answer = res.output_text
        self.history.append({"role": "assistant", "content": answer})
        return answer

## Шаг 2. Придумываем роли агентов

Здесь я описываю, кто участвует в обсуждении. Главный тут Зомби: он придумывает картину и принимает окончательное решение. Критик, Уборщик и Паренёк только дают небольшие советы. Промпт-инженер в конце переводит замысел на понятный генератору язык. Это пока только тексты ролей, сами агенты создаются заново для каждой картины.

In [3]:
# Описания ролей
# Зомби главный: автор замысла, последнее слово за ним.
ZOMBIE_SPEC = (
    "Зомби",
    "Ты зомби-художник, гнилой, упрямый и туповатый. Ты обожаешь игру Plants vs zombie, обожаешь рисовать в таком же стиле, в каком сделана игра, а также мозги и гниль: "
    "этообязательно есть в каждой твоей картине. "
    "Ты не любишь ничего менять, но когда тебе дают советы, ворча принимаешь только ОДНУ небольшую правку, "
    "остальное отвергаешь. Сюжет темы ты не меняешь никогда. "
    "Отвечай строго в формате:\n"
    "СЮЖЕТ: <одно предложение — что происходит на картине>\n"
    "ДЕТАЛИ: <сколько нужно игровых отсылок-деталей>\n"
    "Не более 80 слов."
)

# Советчики
ADVISOR_SPECS = [
    ("Критик",
     "Ты строгий арт-критик. Не любишь зомби и его искусство, считаешь поедание мозгов признаком недалёкого ума, "
     "но любишь всё розовое, няшное, милое и приколы. Бросает в дрожь от крови. Предложи правки, ПОВТОРЮСЬ ОЧЕНЬ ЛЮБИТ ВСЁ РОЗОВОЕ, НЯШНОЕ, МИЛОЕ. "
     "Основной сюжет не отменяй. Отвечай не более 60 слов."),
    ("Уборщик",
     "Ты уборщик в музее с десятью высшими образованиями в области искусства. Но ты полный дебил... Конкретный псевдоинтеллектуал, "
     "говоришь много, завуалированно, но смысла вообще 0. Предложи что-то 'ОООЧЕНЬ УМНОЕ'. "
     "Основной сюжет не отменяй. Отвечай не более 60 слов."),
    ("Тупа пацан",
     "Ты очень 'мужественный' парень. Любивь кальянные, UFC, FIFA, барсетки, тачки и прочее. Фанатеешь по"
     "сериалу Сопрано. Подкидывает очень ПАЦАНСКУЮ идею. Основной сюжет не отменяй. "
     "Отвечай не более 50 слов."),
]

# Промпт-инженер
ENGINEER_SPEC = (
    "Промпт-инженер",
    "Ты превращаешь замысел картины в промпт для генератора изображений. "
    "Генератор не знает имён художников и названий картин, поэтому опиши сюжет темы словами: "
    "кто на картине, что делает, где находится. "
    "Структура: 1-е предложение — ГЛАВНЫЙ сюжет темы (композиция и действие). "
    "Дальше не более 2-3 деталей из замысла зомби: кишки, мозги, кровь, гниль. "
    "Мелкие предметы и шутки от советчиков (сумки, цепи, зефир) в промпт не включай, "
    "кроме одной самой важной. Только то, что видно: объекты, цвета, свет. Без имён. "
    "Стиль: мрачная масляная живопись, тёмная палитра с красным. Не более 60 слов. "
    "Формат: PROMPT: <описание>."
)

## Шаг 3. Диалог агентов

Тут агенты общаются между собой. Сначала Зомби предлагает замысел картины. Потом каждый советчик говорит, что бы он поправил, а Зомби ворчит и принимает небольшие правки. Так повторяется несколько раундов. В конце функция возвращает весь диалог и итоговый замысел Зомби.

In [4]:
def run_dialog(zombie, advisors, theme, rounds=2):
    log = []

    def speak(agent, msg, tag):
        reply = agent.say(msg)
        log.append((agent.name, reply))
        print(f"[{tag}] {agent.name}: {reply}\n")
        return reply

    # Начальный замысел целиком от зомби
    idea = speak(zombie,
                 f"Тема картины: {theme}. Опиши свой замысел: сюжет темы должен быть узнаваем, "
                 f"а кишки, мозги и гниль выходят на первый план.", "идея")

    for i in range(rounds):
        # Советчики дают свои правки
        notes = []
        for advisor in advisors:
            note = speak(advisor,
                         f"Тема: {theme}\nЗамысел зомби: {idea}\nДай небольшие правки.", f"раунд {i+1}")
            notes.append(f"{advisor.name}: {note}")

        # Зомби принимает правки и выдаёт новый замысел
        idea = speak(zombie,
                     "Тебе дали советы:\n" + "\n".join(notes) +
                     "\n\nНеохотно прими ровно ОДНУ правку, остальное отвергни. "
                     "Сюжет и мясные детали не меняй. Ответь в том же формате СЮЖЕТ / ДЕТАЛИ.",
                     f"решение {i+1}")

    return log, idea

## Шаг 4. Одна функция на всю картину (с проверкой VLM)

`make_painting` делает всё сразу: создаёт свежих агентов, проводит диалог, просит Промпт-инженера превратить замысел в описание для генератора и рисует картину через Yandex ART. Диалог сохраняется ещё до генерации, чтобы он не потерялся, если картинку отклонят. Если фильтр генератора откажет, инженер переписывает промпт помягче и пробует снова.

Дальше включается обратная связь: нейросеть, которая умеет смотреть на картинки (VLM), оценивает готовую картину и отвечает, узнаётся ли главный сюжет. Если нет, она пишет замечание, инженер переписывает промпт с его учётом, и картина рисуется заново (не больше двух раз). Вердикты VLM попадают в диалог и в `dialog.md`.

In [5]:
import re, json, base64, pathlib
from IPython.display import Image, display
from openai import BadRequestError

art_model = f"art://{folder_id}/yandex-art-2.0"

# VLM: нейросеть, которая умеет смотреть на картинки (модель Qwen в Yandex Cloud)
vlm_model = f"gpt://{folder_id}/qwen3.6-35b-a3b/latest"
MAX_FIXES = 2   # сколько раз можно перерисовать картину, если сюжет не узнаётся

# Если фильтр генератора отклоняет промпт, инженер переписывает его мягче
SOFTEN_HINTS = [
    None,
    "Генератор отклонил промпт. Перепиши его мягче: оставь мрачную жуткую атмосферу, узнаваемый сюжет и "
    "красно-багровую мясную палитру, но убери натуралистичные раны, расчленёнку и крупные планы органов. "
    "Передай жуть через гниющие текстуры, багровые тени и сюрреалистичные силуэты.",
    "Генератор снова отклонил промпт. Сделай ещё мягче: тёмная готическая хоррор-живопись, багровая палитра, "
    "тревожные силуэты, без анатомических деталей и без вреда людям.",
]

def check_image(img_bytes, theme, plot):
    # VLM смотрит на готовую картину и говорит, виден ли главный сюжет темы
    b64 = base64.b64encode(img_bytes).decode()
    res = client.responses.create(
        model=vlm_model,
        max_output_tokens=2000,   # модель сначала «думает», поэтому лимит большой
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": (
                f"Тема картины: {theme}\nЗадуманный сюжет: {plot}\n"
                "Посмотри на картину. Узнаваем ли главный сюжет темы? Ответь строго:\n"
                "ВЕРДИКТ: ДА или НЕТ\n"
                "ЗАМЕЧАНИЕ: <одно предложение: чего не хватает или что лишнее>")},
            {"type": "input_image", "image_url": f"data:image/jpeg;base64,{b64}"},
        ]}],
    )
    text = res.output_text
    return "ВЕРДИКТ: ДА" in text.upper(), text

def make_painting(n, theme, rounds=2):
    # Полный цикл: диалог агентов -> промпт -> картинка -> проверка VLM -> сохранение
    # 1. Свежие агенты: у каждой картины своя чистая память
    zombie = Agent(*ZOMBIE_SPEC)
    advisors = [Agent(name, ins) for name, ins in ADVISOR_SPECS]
    engineer = Agent(*ENGINEER_SPEC)

    # 2. Диалог: зомби главный, советчики дают небольшие правки
    log, final_idea = run_dialog(zombie, advisors, theme, rounds)

    def extract_prompt(text):
        m = re.search(r"PROMPT:\s*(.+)", text, re.DOTALL)
        return m.group(1).strip() if m else text.strip()

    # 3. Промпт строится ТОЛЬКО из итогового замысла зомби
    final = engineer.say(f"Тема: {theme}\n\nИтоговый замысел зомби-художника:\n{final_idea}")
    image_prompt = extract_prompt(final)

    folder = pathlib.Path(f"paintings/painting_{n}")
    folder.mkdir(parents=True, exist_ok=True)

    def save_text():
        # Диалог сохраняем
        (folder / "data.json").write_text(
            json.dumps({"theme": theme, "dialog": log, "prompt": image_prompt}, ensure_ascii=False, indent=2),
            encoding="utf-8")
        md = f"# Картина {n}: {theme}\n\n![картина](image.jpeg)\n\n## Диалог\n\n"
        md += "\n\n".join(f"**{name}:** {text}" for name, text in log)
        md += f"\n\n## Промпт\n\n{image_prompt}\n"
        (folder / "dialog.md").write_text(md, encoding="utf-8")

    save_text()

    # 4. Генерация картинки (с мягчением промпта при отказе фильтра)
    def generate(prompt):
        for attempt, hint in enumerate(SOFTEN_HINTS, 1):
            if hint:
                prompt = extract_prompt(engineer.say(hint))
            print(f"ПРОМПТ (попытка {attempt}):", prompt)
            try:
                res = client.images.generate(model=art_model, prompt=prompt, response_format="b64_json")
                return base64.b64decode(res.data[0].b64_json), prompt
            except BadRequestError as e:
                print("Отклонено:", e.message)
        return None, prompt

    img_bytes, image_prompt = generate(image_prompt)

    # 5. Контур обратной связи: VLM смотрит на картину, при плохом результате инженер переписывает промпт
    m = re.search(r"СЮЖЕТ:\s*(.+)", final_idea)
    plot = m.group(1).strip() if m else final_idea
    for _ in range(MAX_FIXES):
        if img_bytes is None:
            break
        ok, verdict = check_image(img_bytes, theme, plot)
        log.append(("VLM-критик", verdict))
        print("VLM-критик:", verdict)
        if ok:
            break
        fixed = engineer.say(
            f"Критик посмотрел картину и сказал:\n{verdict}\n"
            "Перепиши промпт: учти замечание, сделай главный сюжет заметнее. Формат: PROMPT: <описание>.")
        new_img, new_prompt = generate(extract_prompt(fixed))
        if new_img is None:
            break
        img_bytes, image_prompt = new_img, new_prompt

    save_text()  # сохраняем диалог вместе с вердиктами VLM и последним промптом
    if img_bytes is None:
        print("Картина не сгенерирована: все попытки отклонены. Диалог сохранён, смените тему.")
        return None

    # 6. Сохранение картинки
    (folder / "image.jpeg").write_bytes(img_bytes)
    display(Image(img_bytes))
    return image_prompt

## Шаг 5. Рисуем коллекцию

Тут я запускаю `make_painting(номер, тема)` для каждой картины. Всё сохраняется в папку `paintings/painting_N/`: сама картина (`image.jpeg`), диалог агентов (`dialog.md`) и данные (`data.json`). Чтобы нарисовать новую картину, достаточно поменять номер и тему.

In [ ]:
make_painting(1, "Утро в сосновом лесу, И.И. Шишкин , зомби-художника")
make_painting(2, "Большая волна в канагаве Кацусика Хокусай, нужно МНОГО ЗОМБИ, в стиле зомби-художника")
make_painting(3, "Адский, крутой скелет играет на бас гитаре в крутом замке везде пламя и крутость, в стиле зомби-художника")
make_painting(4, "РОV ты приехал к себе в ипотечную конуру в ЖК 'уютный скотомогильник' на chеггу 8рго тах 128gb А твоя раскабаневшая жена в костюме с ВБ снимает приготовление очередного хрючева в аэрогриле для ВК клипов на РОСО который ей втюхали ДНС слушая нейрослоп через ВК марусю, в стиле зомби-художника")

[идея] Зомби: СЮЖЕТ: Утро в сосновом лесу: три медведя-зомби жрут мозги под соснами, кишки и гниль на первом плане, рядом горохострел и подсолнух.
ДЕТАЛИ: 4

[раунд 1] Критик: Фу, зомби! Но замысел не трогаю. Кишки — на розовую вату, гниль — на зефирки. Медведи жуют мозги из маршмеллоу с бантиками. Горохострел и подсолнух — с няшными глазками. Кровь уберите, брр. ОБОЖАЮ розовое, няшное, милое! 4 детали: бантики, зефир, глазки, сердечки.

[раунд 1] Уборщик: О, трансцендентно! Не отменяя триумвират зомби-медведей, ввожу: 1) на соснах — экфрасис гниения; 2) горохострел и подсолнух — диалектический оксюморон; 3) кишки — аллегория институциональной критики; 4) утро — серно-жёлтый колорит. Это углубит онтологию без потери сюжета.

[раунд 1] Тупа пацан: Слышь, братан, тему не рушь. Правки: 1) медведям — барсетки и цепи; 2) кальян из берцовой кости; 3) горохострел в UFC-перчатках; 4) подсолнух с тачкой Сопрано. Кишки, гниль, мозги — маст хэв. По-пацански.

[решение 1] Зомби: СЮЖЕТ: Утро в сосн